In [6]:
import sqlite3
import pandas as pd

# 1. Connect to the SQLite database
conn = sqlite3.connect('../m5_forecasting.db')

# 2. Optimized Query 1: Instant Health Check using ROWID index
query_health = """
SELECT 'dim_calendar' AS table_name, MAX(rowid) AS row_count FROM dim_calendar
UNION ALL
SELECT 'dim_sell_prices' AS table_name, MAX(rowid) AS row_count FROM dim_sell_prices
UNION ALL
SELECT 'fact_sales' AS table_name, MAX(rowid) AS row_count FROM fact_sales;
"""

df_health = pd.read_sql(query_health, conn)
print("--- DATABASE HEALTH CHECK ---")
display(df_health)

--- DATABASE HEALTH CHECK ---


,table_name,row_count
0,dim_calendar,1969
1,dim_sell_prices,6841121
2,fact_sales,58327370


In [7]:
# 3. Execute Query 2: Top Selling Products
query_top_products = """
SELECT 
    item_id, 
    SUM(units_sold) AS total_units_sold
FROM fact_sales
GROUP BY item_id
ORDER BY total_units_sold DESC
LIMIT 10;
"""

df_top_products = pd.read_sql(query_top_products, conn)
print("--- TOP 10 SELLING PRODUCTS ---")
display(df_top_products)

--- TOP 10 SELLING PRODUCTS ---


,item_id,total_units_sold
0,FOODS_3_090,1002529
1,FOODS_3_586,920242
2,FOODS_3_252,565299
3,FOODS_3_555,491287
4,FOODS_3_714,396172
5,FOODS_3_587,396119
6,FOODS_3_694,390001
7,FOODS_3_226,363082
8,FOODS_3_202,295689
9,FOODS_3_723,284333


In [16]:
# FIX: Surgically reloading the dim_calendar table with correct mappings
import pandas as pd

# 1. Load the raw calendar CSV (bypassing the processed files)
df_cal = pd.read_csv('../data_raw/calendar.csv')

# 2. Map Kaggle's messy columns to our clean Database Schema
df_cal_mapped = pd.DataFrame({
    'date_id': df_cal['d'],               # The missing link! Maps 'd_1' to date_id
    'calendar_date': df_cal['date'],
    'weekday_name': df_cal['weekday'],
    'weekday_int': df_cal['wday'],
    'month_id': df_cal['month'],
    'year_id': df_cal['year'],
    'event_name_1': df_cal['event_name_1'],
    'event_type_1': df_cal['event_type_1'],
    'event_name_2': df_cal['event_name_2'],
    'event_type_2': df_cal['event_type_2'],
    'snap_CA': df_cal['snap_CA'],
    'snap_TX': df_cal['snap_TX'],
    'snap_WI': df_cal['snap_WI']
})

# 3. Delete the old empty rows and insert the properly mapped data
# Using 'append' preserves your original SQL table structure/data types
cur = conn.cursor()
cur.execute("DELETE FROM dim_calendar;")
conn.commit()

df_cal_mapped.to_sql('dim_calendar', conn, if_exists='append', index=False)

print("--- DIM_CALENDAR FIXED ---")
display(pd.read_sql("SELECT date_id, calendar_date, year_id, month_id FROM dim_calendar LIMIT 5;", conn))

--- DIM_CALENDAR FIXED ---


,date_id,calendar_date,year_id,month_id
0,d_1,2011-01-29,2011,1
1,d_2,2011-01-30,2011,1
2,d_3,2011-01-31,2011,1
3,d_4,2011-02-01,2011,2
4,d_5,2011-02-02,2011,2


In [14]:
# 4. Execute Query 3: Seasonality Analysis (Corrected Schema)
query_seasonality = """
SELECT 
    c.year_id,
    c.month_id,
    SUM(s.units_sold) AS total_units_sold
FROM fact_sales s
INNER JOIN dim_calendar c ON s.date_id = c.date_id
GROUP BY c.year_id, c.month_id
ORDER BY c.year_id, c.month_id;
"""

df_seasonality = pd.read_sql(query_seasonality, conn)
print("--- MONTHLY SALES TRENDS ---")
display(df_seasonality.head(15)) # Displaying the first 15 months

--- MONTHLY SALES TRENDS ---


,year_id,month_id,total_units_sold
0,2011,1,88163
1,2011,2,726375
2,2011,3,763567
3,2011,4,737713
4,2011,5,719562
5,2011,6,753380
6,2011,7,819820
7,2011,8,825694
8,2011,9,813683
9,2011,10,898243
